# Module 1 Lab: Exploring ML in Python

**Practical Machine Learning Foundations**

**Purpose:** Explore numpy and pandas and data visualization

**Date:** 2026-08-24 | **Author:** Nick Garner

Welcome to your first hands-on lab. In Module 1 you learned what machine learning is, the three major ML paradigms, four real-world ML systems, the end-to-end ML workflow, and why every prediction is also an attack surface. In this lab you will put on your data science hat and touch "real data.

Recall the distinction from the first lesson: **AI** is the broad goal of intelligent machines, **ML** is the technique of learning patterns from data, and **data science** is the interdisciplinary practice of extracting insights from data. Everything in this lab (loading, exploring, and visualizing data) is data science work, and it is the essential first stage of every ML project. As was emphasized: data preparation and exploration consume 60 to 80% of total project effort, and one of the most common pitfalls is skipping exploration before modeling.

### What you will do

| Section | Focus |
|---|---|
| Part 1: Intro to pandas | The core data tool of the Python ML stack |
| Part 2: Examining a simple dataset  | Features, labels, and asking the right questions |
| Part 3: Basic visualizations | | Seeing patterns before any model does |

### The scenario

You are a security analyst at a payment company. You have just received a sample of transaction data, and your team is considering building an ML fraud detector (one of the real-world systems from the lessons). Before anyone writes a model, your job is to explore the data. This mirrors the ML workflow: we are working in the **Data Collection & Preparation** stage, well before Model Development.


## Section 0: Colab setup and the Python ML toolkit

If you are reading this in Google Colab, you are already set up: Colab is a free, cloud-hosted Jupyter notebook environment with no local installation required, and the entire Python ML stack comes pre-installed.

**The one shortcut to remember:** `Shift+Enter` runs the current cell and advances to the next one. Try it on the code cell below.

The four libraries of the industry-standard tabular ML stack:

- **NumPy**: fast numerical arrays and math, the foundation everything else builds on
- **pandas**: tabular data manipulation, where you will spend most of your time
- **Matplotlib / Seaborn**: visualization for exploration and evaluation
- **scikit-learn**: a consistent API for classical ML algorithms (`.fit()`, `.predict()`, `.score()`)

In this lab, we use the first three. scikit-learn enters the picture in later modules when we start building models.

Run the standard imports below. These are the same three lines you will type at the top of nearly every notebook in your ML career.


In [ ]:
# Standard imports for the Python ML stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Confirm versions (useful when debugging or reproducing results later)
print("NumPy version:", np.__version__)
print("pandas version:", pd.__version__)

# NOTE: When you execute this cell (shift-enter), the Play symbol in the upper
# left changes while it works.  The output of the print statements will appear
# immediately below the cell.
#
# You should see output similar to this:
# NumPy version: 2.1.3
# pandas version: 2.2.3


---
# Part 1: Intro to pandas

pandas is the tool for working with **tabular data**: data organized in rows and columns, like a spreadsheet or a database table. Nearly every dataset we discussed conceptually so far (labeled transactions for fraud detection, user ratings for recommendations, sensor readings for predictive maintenance, network events for security analytics) arrives as a table, and pandas is how Python works with tables.

pandas has two core structures:

- A **Series** is a single column of values with an index
- A **DataFrame** is a full table: a collection of Series sharing the same index

Let's build each one by hand first, so there is no mystery about what they are.


In [ ]:
# A Series: one labeled column of data
threat_counts = pd.Series(
    [1520, 342, 89, 17],
    index=["phishing", "malware", "ddos", "insider"],
    name="alerts_last_week"
)
threat_counts


A Series pairs each value with an index label. You can grab values by label, and you can do math on the whole Series at once (this is NumPy working underneath):


In [ ]:
print("Phishing alerts:", threat_counts["phishing"])
print("Total alerts:   ", threat_counts.sum())
print()
print("Share of all alerts:")
print((threat_counts / threat_counts.sum() * 100).round(1))


Now a **DataFrame**. Here is a tiny table describing the four real-world ML systems from the lessons, including which paradigm does the core prediction work in each:


In [ ]:
systems = pd.DataFrame({
    "system": ["Recommendation engine", "Fraud detection",
               "Predictive maintenance", "Security analytics (SOC)"],
    "core_paradigm": ["Supervised", "Supervised",
                      "Supervised", "Unsupervised"],
    "core_task": ["Predict clicks and ratings", "Classify fraud vs. legitimate",
                  "Predict remaining useful life", "Flag anomalies in events"],
    "task_type": ["Classification/Regression", "Classification",
                  "Regression", "Anomaly detection"]
})
systems


Notice the pattern: **supervised learning does the core prediction** whenever labeled historical data exists, while **unsupervised** methods work well for exploration and anomaly detection. Production systems are hybrids: recommendation engines also use unsupervised clustering for user segments and reinforcement learning for presentation order, and SOC pipelines layer supervised endpoint classification and RL-assisted response on top of anomaly detection.

### Loading real data

In practice you rarely type data by hand. You load it, most often with `pd.read_csv()`. For this lab we generate a synthetic transaction dataset directly in the notebook so nothing needs to be downloaded, then treat it exactly as if we had loaded it from a file.

Run the cell below. You do not need to understand the generation code, it is just our stand-in for a data source (we set a random seed so everyone gets identical data).


In [ ]:
# Generate the lab dataset: 5,000 payment transactions, ~1.5% fraudulent.
# In a real project this cell would simply be:
#     df = pd.read_csv("transactions.csv")
rng = np.random.default_rng(42)
n = 5000
n_fraud = 75    # fraud is rare, roughly 1.5% of transactions

# Legitimate transactions: modest amounts, mostly daytime, familiar merchants
legit_amount = np.round(rng.lognormal(mean=3.4, sigma=0.9, size=n - n_fraud), 2)
legit_hour = rng.normal(loc=14, scale=4.5, size=n - n_fraud).astype(int) % 24
legit_dist = np.round(np.abs(rng.normal(loc=8, scale=12, size=n - n_fraud)), 1)

# Fraudulent transactions: larger amounts, late night, far from home
fraud_amount = np.round(rng.lognormal(mean=5.1, sigma=0.8, size=n_fraud), 2)
fraud_hour = rng.normal(loc=3, scale=2.5, size=n_fraud).astype(int) % 24
fraud_dist = np.round(np.abs(rng.normal(loc=400, scale=300, size=n_fraud)), 1)

merchants = ["grocery", "online_retail", "gas_station", "restaurant",
             "electronics", "travel", "entertainment"]

df = pd.DataFrame({
    "transaction_id": np.arange(1, n + 1),
    "amount": np.concatenate([legit_amount, fraud_amount]),
    "hour_of_day": np.concatenate([legit_hour, fraud_hour]),
    "distance_from_home_km": np.concatenate([legit_dist, fraud_dist]),
    "merchant_category": rng.choice(merchants, size=n,
                                    p=[.25, .22, .15, .15, .09, .07, .07]),
    "day_of_week": rng.choice(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"], size=n),
    "is_fraud": np.concatenate([np.zeros(n - n_fraud, dtype=int),
                                np.ones(n_fraud, dtype=int)])
})

# Shuffle rows so fraud is not conveniently sorted at the bottom
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("Dataset created:", df.shape[0], "rows and", df.shape[1], "columns")


### First look: head, tail, and shape

The universal first move with any new DataFrame is `.head()`: show the first five rows.


In [ ]:
df.head()


In [ ]:
# .tail() shows the last rows, .shape gives (rows, columns) as a tuple
print("Last 3 rows:")
display(df.tail(3))
print("Shape (rows, columns):", df.shape)


In [ ]:
df.sort_values(by="transaction_id")


### Structure and summary: info() and describe()

`.info()` answers three questions at once: how many rows, what type is each column, and are any values missing. Data quality problems (missing values, wrong types, duplicates) are the norm in real data, and this is your first screening tool. Module 3 covers the full cleaning toolkit; today we just look.


In [ ]:
df.info()


Read the output: 5,000 entries, no missing values in any column (each shows 5000 non-null), numeric columns stored as integers or floats, and text columns stored as `object`. Real datasets are rarely this clean, which is exactly why you check.

`.describe()` computes summary statistics for the numeric columns:


In [ ]:
df.describe().round(2)


Two things worth noticing:

1. **`amount`**: the mean is noticeably higher than the median (the 50% row). A few very large transactions pull the average up. Statisticians call this a skewed distribution, and you will see its shape in Part 3.
2. **`is_fraud`**: the mean is about 0.015. Since the column is 0 or 1, the mean is the fraud rate: roughly 1.5% of transactions. This is the **class imbalance** problem, we will return to it in Part 2.

### Selecting columns

Square brackets with a column name give you a single column back as a Series. A list of names gives you a smaller DataFrame.


In [ ]:
# One column: a Series
amounts = df["amount"]
print(type(amounts))
print(amounts.head(3))
print()

# Several columns: a DataFrame
df[["amount", "merchant_category", "is_fraud"]].head(3)


### Filtering rows

Filtering is the pandas skill you will use most. Write a condition, get back only the rows where it is true. Conditions can be combined with `&` (and) and `|` (or), with each condition wrapped in parentheses.


In [ ]:
# All transactions over $500
large = df[df["amount"] > 500]
print("Transactions over $500:", len(large))

# Late-night AND far from home: the classic suspicious combination
suspicious = df[(df["hour_of_day"] < 5) & (df["distance_from_home_km"] > 100)]
print("Late night, far from home:", len(suspicious))

# What fraction of those suspicious transactions are actually fraud?
print("Fraud rate within that slice:",
      round(suspicious["is_fraud"].mean() * 100, 1), "%")
print("Fraud rate overall:          ",
      round(df["is_fraud"].mean() * 100, 1), "%")


That last comparison is a small but genuine insight: a hand-written filter (late night and far from home) concentrates fraud far above the base rate. This is exactly the kind of **rule-based logic** that powered early AI systems: explicit, hand-coded if/then conditions. It works, but it only catches patterns a human already thought of. The promise of ML, and the reason it displaced pure rule-based approaches, is discovering the patterns nobody thought to write down, across many variables at once (the **pattern discovery** advantage). Also remember the caveat: fraudsters are **adversarial actors** who adapt, so any fixed rule eventually gets learned and evaded, which is the concept-drift arms race in miniature.

### Counting categories and sorting

`value_counts()` counts how many times each value appears. `.sort_values()` orders rows by a column.


In [ ]:
# How are transactions spread across merchant categories?
print(df["merchant_category"].value_counts())
print()

# The label column: how imbalanced are our classes?
print(df["is_fraud"].value_counts())


In [ ]:
counts = df["merchant_category"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
counts.plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis()          # largest on top
ax.set_title("Transactions by merchant category")
ax.set_xlabel("count")
ax.set_ylabel("")

for i, v in enumerate(counts):
    ax.text(v + 15, i, str(v), va="center")

plt.tight_layout()
plt.show()

In [ ]:
# The five largest transactions, biggest first
df.sort_values("amount", ascending=False).head(5)


### Creating a new column

New columns are created by assignment. This is your first tiny taste of **feature engineering**: transforming raw data into more meaningful signals. Recall the example from the slides: a raw timestamp is not very useful, but "is this during typical active hours" captures a pattern that matters.


In [ ]:
# A simple engineered feature: flag late-night transactions
df["is_late_night"] = (df["hour_of_day"] < 5).astype(int)

# groupby: split rows into groups and summarize each group
df.groupby("is_late_night")["is_fraud"].mean().round(4)


`groupby` split the data into late-night vs. daytime groups and computed the fraud rate for each. The late-night fraud rate is dramatically higher. A model would learn this pattern on its own, but as an analyst you just confirmed the signal exists, and you did it before writing a single line of model code.

**Part 1 recap.** You can now create Series and DataFrames, load and inspect data (`head`, `tail`, `shape`, `info`, `describe`), select columns, filter rows, count categories, sort, engineer a simple column, and summarize by group. Later lessons go deeper into selecting, filtering, and grouping; this is the foundation.


---
# Part 2: Examining a simple dataset

You know the mechanics. Now think like an ML practitioner and ask the questions Module 1 taught you to ask about any dataset.

### Question 1: What are the features, and what is the label?

From the lessons: supervised learning learns from **labeled examples**, input-output pairs. The inputs are called **features** and the correct answers are called **labels**.


In [ ]:
feature_cols = ["amount", "hour_of_day", "distance_from_home_km",
                "merchant_category", "day_of_week", "is_late_night"]
label_col = "is_fraud"

print("FEATURES (the inputs a model would learn from):")
for c in feature_cols:
    print("  ", c)
print()
print("LABEL (the answer we want to predict):", label_col)


Note what is **not** a feature: `transaction_id`. It is just a row identifier with no predictive meaning.

### Question 2: Which ML paradigm fits?

Run this dataset through the decision tree the lesson:

1. **Do we have labeled data?** Yes: every transaction is marked fraud or legitimate. **Supervised learning.**
2. **Is the label a category or a number?** A category (fraud / not fraud), and specifically two categories. **Binary classification**, just like the spam filter and fraud examples from the slides. If we were instead predicting a dollar amount of loss, that would be **regression** (a continuous value).

And the counterfactuals matter just as much:

- **No labels?** If our transactions arrived unlabeled, we would turn to **unsupervised learning**: clustering to find natural transaction groups, or anomaly detection to model normal behavior and flag deviations, exactly how SOC network analytics catches novel attacks with no known signature. Techniques like K-Means, DBSCAN, and PCA.
- **Sequential decisions with feedback over time?** If we were adaptively tuning blocking rules against an evolving attacker, that would be **reinforcement learning**: an agent taking actions in an environment, guided by rewards. Think adaptive firewall management from the scenarios.
- **Hybrid reality:** a production fraud system might use all three, unsupervised methods to surface new patterns, supervised classification to make the approve/block decision, and RL to adapt thresholds. Most production ML systems combine paradigms.

### Question 3: How balanced are the classes?


In [ ]:
counts = df["is_fraud"].value_counts()
pct = df["is_fraud"].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    "count": counts,
    "percent": pct.round(2)
})
summary.index = ["legitimate (0)", "fraud (1)"]
summary


About 98.5% legitimate, 1.5% fraud. Real payment networks are even more extreme (roughly 0.1% fraud). This is **class imbalance**, recall:

> **Accuracy is useless here.** A "model" that predicts *legitimate* for every single transaction scores about 98.5% accuracy and catches zero fraud.

Prove it to yourself:


In [ ]:
# The do-nothing baseline: predict "legitimate" for everything
always_legit_accuracy = (df["is_fraud"] == 0).mean()
print("Accuracy of predicting 'legitimate' every time:",
      round(always_legit_accuracy * 100, 2), "%")
print("Fraud caught by this 'model': 0 out of", df["is_fraud"].sum())


This is why we introduced metrics beyond accuracy: **precision** (of the transactions flagged as fraud, how many really were?), **recall** (of all real fraud, how many did we catch?), **F1** (the balance of both), and **AUC-ROC** (how well the model ranks fraud above legitimate). It is also why the lesson lists *wrong metric* and *ignoring class imbalance* among the classic evaluation mistakes, and why every model should beat a **baseline** like the one you just computed. Costs are asymmetric too: a missed fraud costs far more than a false alarm, and in security analytics a high false positive rate creates **alert fatigue**, where analysts drown in noise and start ignoring alerts. Metrics get their full treatment in Module 4; for now, carry the intuition.

### Question 4: Can we trust this data?

Put on the security hat: every prediction is a target, and the attack surface starts at the data.

- **Data poisoning**: our labels record which transactions were *confirmed* fraud. A fraudster who keeps their transactions from ever being labeled fraud is poisoning the training set, teaching the future model that their pattern is legitimate. Microsoft's Tay chatbot fell to coordinated poisoning within hours. Bad labels are also just a quality problem: garbage labels, garbage model.
- **Adversarial inputs**: every fraudulent transaction is an attempt to look legitimate to the model at decision time, the same idea as stickers fooling a vision system or perturbations fooling a diagnostic model.
- **Model inversion, extraction, and membership inference**: once a model is deployed, attackers can query it to reconstruct private training data, clone its logic, or test whether a specific record was in the training set. Rate limiting and query controls at deployment exist for exactly this reason.
- **Drift**: fraud tactics evolve, so the patterns in this snapshot decay. **Data drift** shifts the input distributions, **concept drift** changes the relationship between features and fraud. A model launched at 95% recall can quietly degrade within months, which is why monitoring and retraining are workflow stages, not afterthoughts.
- **Data problems** apply too: is this sample **biased** (does it represent real traffic)? Is it **stale**? And is there **leakage**, any feature that would not be available at prediction time? (Imagine a column called `chargeback_filed`: fantastic validation scores, useless in production, because the outcome leaked into the features.)

You cannot fix any of that today, but noticing it is the job. Security is not a module you bolt on later, it is a lens you apply from the very first `head()`.


---
# Part 3: Basic visualizations

Numbers summarize; pictures reveal. Visualization is a core data science skill and the heart of exploratory analysis. Remember the pitfall: never skip exploration before modeling. These four plots are the workhorses.

### Plot 1: Histogram, the shape of one variable


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["amount"], bins=60, color="steelblue", edgecolor="white")
plt.title("Distribution of Transaction Amounts")
plt.xlabel("Amount ($)")
plt.ylabel("Number of transactions")
plt.show()


A long right tail: most transactions are small, a few are very large. This is the skew that `.describe()` hinted at when the mean exceeded the median. Distributions like this often get transformed during preprocessing (Module 3).

### Plot 2: Bar chart, comparing categories

Bar charts show counts or values per category. First, transactions per merchant category. Then the plot that matters most for this dataset: the class balance.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: transactions by merchant category
df["merchant_category"].value_counts().plot(
    kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Transactions by Merchant Category")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

# Right: class imbalance, seen rather than described
df["is_fraud"].value_counts().plot(
    kind="bar", ax=axes[1], color=["steelblue", "firebrick"])
axes[1].set_title("Class Balance: Legitimate vs. Fraud")
axes[1].set_xticklabels(["Legitimate", "Fraud"], rotation=0)
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


The right-hand plot is class imbalance made visible: the fraud bar is barely a sliver. One glance communicates what a paragraph of statistics struggles to, which is why visualization is also a **communication** tool for stakeholders, not just an exploration tool.

### Plot 3: Scatter plot, the relationship between two variables

Now the most interesting plot of the lab. Each point is one transaction, positioned by hour of day and distance from home, colored by its label.


In [ ]:
legit = df[df["is_fraud"] == 0]
fraud = df[df["is_fraud"] == 1]

plt.figure(figsize=(9, 5))
plt.scatter(legit["hour_of_day"], legit["distance_from_home_km"],
            s=12, alpha=0.3, color="steelblue", label="Legitimate")
plt.scatter(fraud["hour_of_day"], fraud["distance_from_home_km"],
            s=30, alpha=0.9, color="firebrick", label="Fraud")
plt.title("Hour of Day vs. Distance from Home")
plt.xlabel("Hour of day")
plt.ylabel("Distance from home (km)")
plt.legend()
plt.show()


The fraud points cluster in a distinct region: late-night hours, large distances. **This visible separation is exactly what a supervised classifier learns to exploit.** When Module 4 introduces decision boundaries, picture this plot: the model's job is to draw a line (or curve) separating red from blue and then generalize it to transactions it has never seen. Generalizing to new data, not memorizing old data, is the entire point of ML, and the gap between the two is the overfitting problem.

Two more Module 1 ideas live in this picture:

- **Adversarial evasion**: a smart fraudster who knew this model would commit fraud at 2 PM close to home, deliberately placing their point inside the blue cloud. The model's learned pattern is also its blind spot.
- **Anomaly detection**: even with all labels hidden, most red points would still look unusual against the dense blue mass. That is the unsupervised route: model normal, flag deviations, the same principle behind user behavior analytics and network anomaly detection in the SOC stack.

### Plot 4: Line plot, values in sequence

Line plots show how a value changes across an ordered axis, most often time. Here: average transaction amount by hour of day.


In [ ]:
hourly = df.groupby("hour_of_day")["amount"].mean()

plt.figure(figsize=(9, 4))
plt.plot(hourly.index, hourly.values, marker="o", color="steelblue")
plt.title("Average Transaction Amount by Hour of Day")
plt.xlabel("Hour of day")
plt.ylabel("Average amount ($)")
plt.grid(alpha=0.3)
plt.show()


The overnight hours spike, dragged upward by the large fraudulent transactions concentrated there. In production, time-ordered views like this are the backbone of **monitoring**: they are how teams spot data drift and model degradation before customers do. And when the sequence itself carries the signal, as in predictive maintenance where a 30-day vibration trend predicts a bearing failure while a single reading says nothing, specialized time-series techniques take over: sliding-window features as the simple baseline, and sequence models (RNNs, LSTMs, TCNs) when degradation patterns are gradual.


---
# Try it yourself

Three short exercises using only skills from this lab. Solutions are at the bottom of the notebook.

**Exercise 1.** What fraction of `travel` transactions are fraudulent? How does it compare to the overall fraud rate? (Hint: filter, then take the mean of `is_fraud`.)

**Exercise 2.** Using `groupby`, find the average transaction `amount` for fraudulent vs. legitimate transactions.

**Exercise 3.** Make a histogram of `distance_from_home_km` for **legitimate transactions only**. What shape does it have?


In [ ]:
# Exercise workspace: write your answers here





---
# Wrap-up: where this lab fits

Map today's work onto the end-to-end ML workflow:

| Workflow stage | This lab |
|---|---|
| 1. Problem definition | Given: detect fraud, a binary classification problem with a measurable target |
| 2. Data collection & preparation | **You were here**: loading, inspecting, exploring, one engineered feature |
| 3. Model development | Module 4: algorithms, training, train/test splits, hyperparameters |
| 4. Model evaluation | Module 4: precision, recall, F1, baselines, honest splits |
| 5. Deployment | Later modules: validation, authentication, rate limiting, rollback plans |
| 6. Monitoring & maintenance | Later modules: drift detection, retraining, security monitoring |

Most ML failures are process failures, not algorithm failures, and the anti-failure checklist starts here: verify ML is the right tool, invest in data quality, explore before you model, and keep the security lens on from day one. ML delivers its advantages (scale, speed, pattern discovery, continuous improvement, automation) only when this groundwork is done honestly. That is also why ML skills are in such demand across roles, from data scientists to security professionals: the fundamentals you practiced today are the same ones production teams use, just at a larger scale.

**Next up:** Python ML tooling in depth, Jupyter and Colab power features, and serious pandas for data analysis.


---
## Exercise solutions


In [ ]:
# Exercise 1: fraud rate within travel transactions vs. overall
travel = df[df["merchant_category"] == "travel"]
print("Travel fraud rate: ", round(travel["is_fraud"].mean() * 100, 2), "%")
print("Overall fraud rate:", round(df["is_fraud"].mean() * 100, 2), "%")


In [ ]:
# Exercise 2: average amount, fraud vs. legitimate
df.groupby("is_fraud")["amount"].mean().round(2)


In [ ]:
# Exercise 3: distance histogram for legitimate transactions only
legit_only = df[df["is_fraud"] == 0]

plt.figure(figsize=(8, 4))
plt.hist(legit_only["distance_from_home_km"], bins=50,
         color="steelblue", edgecolor="white")
plt.title("Distance from Home, Legitimate Transactions Only")
plt.xlabel("Distance from home (km)")
plt.ylabel("Number of transactions")
plt.show()
# Shape: strongly right-skewed, most purchases happen close to home
